# Ball Path Characteristics (Abhi)



This notebook builds pre-release 3D ball trajectories per shot using phase coordinates (`PreHitch -> Hitch -> PostHitch -> Release`) and phase times normalized by `timeAtStartofMovement`.



Key behaviors:

- Interpolating spline (no smoothing) fit per shot in 3D over normalized time

- Every trajectory is re-centered to start at the origin

- Interactive Plotly 3D visualization with filters for player and shot type

- Velocity-based coloring along each interpolated curve

In [2]:
import warnings

from pathlib import Path



import numpy as np

import pandas as pd

import plotly.graph_objects as go

from plotly.colors import sample_colorscale

from scipy.interpolate import PchipInterpolator



try:

    import ipywidgets as widgets

    from IPython.display import display

    HAS_WIDGETS = True

except Exception:

    HAS_WIDGETS = False



warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 200)

In [3]:
DATA_PATH = Path('..') / 'capstone2026v2.csv'

df = pd.read_csv(DATA_PATH)

print(f'Loaded rows: {len(df):,}; columns: {df.shape[1]}')


def pick_col(candidates, columns):

    return next((c for c in candidates if c in columns), None)



columns = set(df.columns)



PLAYER_CANDS = ['Name', 'Player', 'Player.Name']

SHOT_TYPE_CANDS = ['Shot.Type', 'ShotType', 'Shot Type']

TIME_START_CANDS = ['timeAtStartofMovement', 'timeAtStartOfMovement', 'TimeAtStartOfMovement']



player_col = pick_col(PLAYER_CANDS, columns)

shot_type_col = pick_col(SHOT_TYPE_CANDS, columns)

t_start_col = pick_col(TIME_START_CANDS, columns)



phase_order = ['PreHitch', 'Hitch', 'PostHitch', 'Release']

phase_xyz_cols = {

    'PreHitch': {'x': 'BallXPreHitch', 'y': 'BallYPreHitch', 'z': 'BallZPreHitch'},

    'Hitch': {'x': 'BallXHitch', 'y': 'BallYHitch', 'z': 'BallZHitch'},

    'PostHitch': {'x': 'BallXPostHitch', 'y': 'BallYPostHitch', 'z': 'BallZPostHitch'},

    'Release': {'x': 'BallXRelease', 'y': 'BallYRelease', 'z': 'BallZRelease'},

}

# Use strictly ordered event times to avoid duplicate-time spline instability.
phase_time_cols = {

    'PreHitch': 'timeAtStartofMovement',

    'Hitch': 'timeAtStartofHitch',

    'PostHitch': 'timeAtEndofHitch',

    'Release': 'timeAtRelease',

}

phase_vel_cols = {

    'PreHitch': 'BallVeloStart',

    'Hitch': 'BallVeloPreHitch',

    'PostHitch': 'BallVeloPostHitch',

    'Release': 'BallVeloRelease',

}

# Standardization strategy applied AFTER spline fitting.
# - 'none': no post-fit scaling.
# - 'player_axis_std': divide x/y/z by each player's axis-specific std.
POST_STANDARDIZATION = 'player_axis_std'



required_cols = []

required_cols.extend([v for p in phase_order for v in phase_xyz_cols[p].values()])

required_cols.extend([phase_time_cols[p] for p in phase_order])

required_cols.extend([phase_vel_cols[p] for p in phase_order])

required_cols.extend([player_col, shot_type_col, t_start_col])

required_cols = [c for c in required_cols if c is not None]



missing = [c for c in required_cols if c not in df.columns]

if missing:

    raise ValueError(f'Missing required columns: {missing}')



print('Using columns:')

print(f'  player: {player_col}')

print(f'  shot type: {shot_type_col}')

print(f'  movement start time: {t_start_col}')

print('  shooter proxy location: BallXPreHitch, BallYPreHitch, BallZPreHitch')

print('  basket is treated as (0, 0) in XY and each shot is rotated so pre-hitch->basket points +Y')

print(f'  post-fit standardization mode: {POST_STANDARDIZATION}')

Loaded rows: 25,868; columns: 264
Using columns:
  player: Name
  shot type: Shot.Type
  movement start time: timeAtStartofMovement
  shooter proxy location: BallXPreHitch, BallYPreHitch, BallZPreHitch
  basket is treated as (0, 0) in XY and each shot is rotated so pre-hitch->basket points +Y
  post-fit standardization mode: player_axis_std


In [4]:
# Build phase-level table per shot and normalize to movement start + pre-hitch-centered frame

shots = []

skip_counts = {
    'missing_values': 0,
    'time_not_monotonic': 0,
    'not_enough_unique_times': 0,
}

orientation_counts = {'rotated_to_basket_axis': 0, 'basket_at_origin': 0}



for idx, row in df.iterrows():

    x = np.array([row[phase_xyz_cols[p]['x']] for p in phase_order], dtype=float)

    y = np.array([row[phase_xyz_cols[p]['y']] for p in phase_order], dtype=float)

    z = np.array([row[phase_xyz_cols[p]['z']] for p in phase_order], dtype=float)

    t = np.array([row[phase_time_cols[p]] for p in phase_order], dtype=float)

    v = np.array([row[phase_vel_cols[p]] for p in phase_order], dtype=float)

    t0 = float(row[t_start_col])



    # Use pre-hitch ball position as the shooter proxy in ball-coordinate space.
    px = float(x[0])
    py = float(y[0])
    pz = float(z[0])



    if not np.isfinite(np.r_[x, y, z, t, v, t0, px, py, pz]).all():

        skip_counts['missing_values'] += 1

        continue



    t_norm = t - t0

    # Require strict temporal ordering to keep interpolation stable.
    if np.any(np.diff(t_norm) <= 0):

        skip_counts['time_not_monotonic'] += 1

        continue



    if np.unique(t_norm).size < 4:

        skip_counts['not_enough_unique_times'] += 1

        continue



    # Translate into pre-hitch-centered frame.
    x = x - px

    y = y - py

    z = z - pz

    # Rotate XY so the pre-hitch->basket vector aligns with +Y for all shots.
    # Basket is assumed at (0, 0) in this coordinate system.
    to_basket_x = -px
    to_basket_y = -py
    basket_dist = float(np.hypot(to_basket_x, to_basket_y))

    if basket_dist > 1e-8:
        theta = np.arctan2(to_basket_x, to_basket_y)
        c = np.cos(theta)
        s = np.sin(theta)
        x_rot = c * x - s * y
        y_rot = s * x + c * y
        x, y = x_rot, y_rot
        orientation_counts['rotated_to_basket_axis'] += 1
    else:
        orientation_counts['basket_at_origin'] += 1



    shots.append({

        'shot_id': int(idx),

        'player': row[player_col],

        'shot_type': row[shot_type_col],

        'phase': phase_order,

        't_phase': t_norm,

        't_phase_safe': t_norm,

        'x_phase': x,

        'y_phase': y,

        'z_phase': z,

        'v_phase': v,

    })



shots_df = pd.DataFrame(shots)

print(f'Valid shots: {len(shots_df):,}')

print('Skipped shots:', skip_counts)
print('Orientation ops:', orientation_counts)

Valid shots: 25,805
Skipped shots: {'missing_values': 0, 'time_not_monotonic': 63, 'not_enough_unique_times': 0}
Orientation ops: {'rotated_to_basket_axis': 25805, 'basket_at_origin': 0}


In [5]:
def interpolate_shot(row, n_samples=80, max_abs_coord=200.0):

    t = np.asarray(row['t_phase_safe'], dtype=float)

    x = np.asarray(row['x_phase'], dtype=float)

    y = np.asarray(row['y_phase'], dtype=float)

    z = np.asarray(row['z_phase'], dtype=float)

    v = np.asarray(row['v_phase'], dtype=float)



    t_dense = np.linspace(t.min(), t.max(), int(n_samples))

    # Shape-preserving interpolation avoids spline overshoot with sparse points.
    fx = PchipInterpolator(t, x)
    fy = PchipInterpolator(t, y)
    fz = PchipInterpolator(t, z)

    x_dense = fx(t_dense)
    y_dense = fy(t_dense)
    z_dense = fz(t_dense)

    # Safety fallback if any shot still produces unrealistic coordinates.
    if np.max(np.abs(np.r_[x_dense, y_dense, z_dense])) > max_abs_coord:
        x_dense = np.interp(t_dense, t, x)
        y_dense = np.interp(t_dense, t, y)
        z_dense = np.interp(t_dense, t, z)



    v_dense = np.interp(t_dense, t, v)



    return pd.DataFrame({

        'shot_id': int(row['shot_id']),

        'player': row['player'],

        'shot_type': row['shot_type'],

        't_norm': t_dense,

        'x': x_dense,

        'y': y_dense,

        'z': z_dense,

        'ball_velocity': v_dense,

    })



traj_parts = [interpolate_shot(row, n_samples=80) for _, row in shots_df.iterrows()]

traj_df = pd.concat(traj_parts, ignore_index=True) if traj_parts else pd.DataFrame()

# Standardize x/y/z within player AFTER interpolation.
if POST_STANDARDIZATION == 'player_axis_std' and not traj_df.empty:
    for axis in ['x', 'y', 'z']:
        player_scale = traj_df.groupby('player')[axis].transform('std')
        global_scale = float(traj_df[axis].std())
        player_scale = player_scale.fillna(global_scale)
        player_scale = player_scale.mask(player_scale.abs() < 1e-8, global_scale)
        if np.isfinite(global_scale) and global_scale > 1e-8:
            traj_df[axis] = traj_df[axis] / player_scale

    print('Applied post-fit player-wise axis standardization for x, y, z.')



print(f'Trajectory samples: {len(traj_df):,}')

if not traj_df.empty:

    starts = (traj_df.sort_values(['shot_id', 't_norm']).groupby('shot_id').first()[['x', 'y', 'z']])

    start_radius = np.sqrt((starts ** 2).sum(axis=1))

    coord_abs_max = float(traj_df[['x', 'y', 'z']].abs().to_numpy().max())

    print('Player-centered start distance stats (mean / median / max):',

          f"{start_radius.mean():.4f} / {start_radius.median():.4f} / {start_radius.max():.4f}")

    print(f'Max absolute coordinate after interpolation: {coord_abs_max:.4f}')

Applied post-fit player-wise axis standardization for x, y, z.
Trajectory samples: 2,064,400
Player-centered start distance stats (mean / median / max): 0.0000 / 0.0000 / 0.0000
Max absolute coordinate after interpolation: 18.4082


In [11]:
def velocity_to_color(values, cmin, cmax, colorscale='Turbo'):

    if cmax <= cmin:

        return ['rgb(128,128,128)'] * len(values)

    normed = np.clip((np.asarray(values) - cmin) / (cmax - cmin), 0.0, 1.0)

    return [sample_colorscale(colorscale, float(v))[0] for v in normed]



def make_trajectory_figure(data, max_shots=120, title_suffix=''):

    fig = go.Figure()

    if data.empty:

        fig.update_layout(title='No data after filters')

        return fig



    vel_min = float(data['ball_velocity'].min())

    vel_max = float(data['ball_velocity'].max())



    shot_ids = data['shot_id'].drop_duplicates().tolist()

    if len(shot_ids) > max_shots:

        shot_ids = shot_ids[:max_shots]

        data = data[data['shot_id'].isin(shot_ids)]



    for shot_id, g in data.groupby('shot_id', sort=False):

        g = g.sort_values('t_norm')

        x = g['x'].to_numpy()

        y = g['y'].to_numpy()

        z = g['z'].to_numpy()

        vel = g['ball_velocity'].to_numpy()



        seg_colors = velocity_to_color((vel[:-1] + vel[1:]) / 2.0, vel_min, vel_max, colorscale='Turbo')

        for i in range(len(x) - 1):

            fig.add_trace(go.Scatter3d(

                x=[x[i], x[i + 1]],

                y=[y[i], y[i + 1]],

                z=[z[i], z[i + 1]],

                mode='lines',

                line=dict(color=seg_colors[i], width=5),

                hoverinfo='skip',

                showlegend=False,

            ))



        fig.add_trace(go.Scatter3d(

            x=x,

            y=y,

            z=z,

            mode='markers',

            marker=dict(

                size=2,

                color=vel,

                colorscale='Turbo',

                cmin=vel_min,

                cmax=vel_max,

                showscale=False,

            ),

            customdata=np.stack([g['player'], g['shot_type'], g['t_norm']], axis=-1),

            hovertemplate=(

                'Shot ID: %{text}<br>'

                'Player: %{customdata[0]}<br>'

                'Shot Type: %{customdata[1]}<br>'

                't_norm: %{customdata[2]:.4f}s<br>'

                'Velocity: %{marker.color:.3f}<br>'

                'x,y,z: (%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>'

            ),

            text=[shot_id] * len(g),

            showlegend=False,

        ))



    fig.add_trace(go.Scatter3d(

        x=[None], y=[None], z=[None],

        mode='markers',

        marker=dict(

            size=0.1,

            color=[vel_min],

            colorscale='Turbo',

            cmin=vel_min,

            cmax=vel_max,

            showscale=True,

            colorbar=dict(title='Ball Velocity')

        ),

        hoverinfo='skip',

        showlegend=False

    ))



    fig.update_layout(

        title=f'Pre-Release Ball Trajectories {title_suffix}'.strip(),

        scene=dict(

            xaxis_title='X (origin-shifted)',

            yaxis_title='Y (origin-shifted)',

            zaxis_title='Z (origin-shifted)',

            aspectmode='data',

        ),

        height=820,

        margin=dict(l=0, r=0, t=50, b=0),

    )

    return fig

In [45]:
all_players = sorted(traj_df['player'].dropna().astype(str).unique().tolist()) if not traj_df.empty else []

all_shot_types = sorted(traj_df['shot_type'].dropna().astype(str).unique().tolist()) if not traj_df.empty else []



print(f'Players: {len(all_players)} | Shot types: {len(all_shot_types)}')



def filter_trajectories(data, players=None, shot_types=None):

    if data.empty:

        return data

    out = data

    if players and 'All' not in players:

        out = out[out['player'].astype(str).isin(players)]

    if shot_types and 'All' not in shot_types:

        out = out[out['shot_type'].astype(str).isin(shot_types)]

    return out



if HAS_WIDGETS:

    player_options = ['All'] + all_players

    shot_options = ['All'] + all_shot_types



    player_select = widgets.SelectMultiple(

        options=player_options,

        value=('All',),

        description='Player',

        rows=min(12, max(6, len(player_options))),

        layout=widgets.Layout(width='360px')

    )

    shot_select = widgets.SelectMultiple(

        options=shot_options,

        value=('All',),

        description='Shot Type',

        rows=min(10, max(5, len(shot_options))),

        layout=widgets.Layout(width='360px')

    )

    max_shots_slider = widgets.IntSlider(

        value=120, min=20, max=500, step=20,

        description='Max Shots',

        continuous_update=False,

        layout=widgets.Layout(width='360px')

    )



    out = widgets.Output()



    def redraw(*_):

        out.clear_output(wait=True)

        f = filter_trajectories(

            traj_df,

            players=list(player_select.value),

            shot_types=list(shot_select.value),

        )

        suffix = f'| shots shown: {f["shot_id"].nunique()}' if not f.empty else ''

        fig = make_trajectory_figure(f, max_shots=max_shots_slider.value, title_suffix=suffix)

        with out:

            fig.show()



    player_select.observe(redraw, names='value')

    shot_select.observe(redraw, names='value')

    max_shots_slider.observe(redraw, names='value')



    controls = widgets.HBox([player_select, shot_select])

    display(widgets.VBox([controls, max_shots_slider, out]))

    redraw()

else:

    print('ipywidgets not available. Using a default all-data figure fallback...')

    fig = make_trajectory_figure(traj_df, max_shots=120)

    fig.show()

Players: 165 | Shot types: 2


## Catch-and-Shoot Clustering with Geomstats

This section clusters normalized catch-and-shoot trajectories using geometric shape alignment.

Pipeline:
1. Filter to catch-and-shoot shots.
2. Build one fixed-length 3D trajectory per shot.
3. Normalize velocity **within each shot** (to reduce distance-driven magnitude effects).
4. Align curves with Geomstats (SRV metric with rotation + reparametrization quotient).
5. Cluster aligned curves and inspect cluster-average trajectories.

### Notes on Velocity Normalization

Velocity is normalized **within each shot** as a z-score over that shot's sampled trajectory points before clustering features are formed.

This helps remove raw speed magnitude effects tied to distance, while still preserving *how* velocity changes through the shot path.

In [12]:
# Visualize mean aligned trajectory per cluster (catch-and-shoot)
# using velocity gradient along each centroid curve.
cluster_means = []
for c in sorted(curves_df['cluster'].unique()):
    idx = curves_df.index[curves_df['cluster'] == c].to_numpy()
    mean_curve = aligned_curves[idx].mean(axis=0)
    mean_vel = vel_norm[idx].mean(axis=0)
    cluster_means.append((c, mean_curve, mean_vel, len(idx)))

vel_all = np.concatenate([mvel for _, _, mvel, _ in cluster_means])
vel_min = float(np.min(vel_all))
vel_max = float(np.max(vel_all))

fig_cluster = go.Figure()

for c, mean_curve, mean_vel, n_c in cluster_means:
    x = mean_curve[:, 0]
    y = mean_curve[:, 1]
    z = mean_curve[:, 2]

    # Segment-wise color from average velocity between adjacent points.
    seg_vel = (mean_vel[:-1] + mean_vel[1:]) / 2.0
    seg_colors = velocity_to_color(seg_vel, vel_min, vel_max, colorscale='Turbo')

    for i in range(len(x) - 1):
        fig_cluster.add_trace(go.Scatter3d(
            x=[x[i], x[i + 1]],
            y=[y[i], y[i + 1]],
            z=[z[i], z[i + 1]],
            mode='lines',
            line=dict(width=9, color=seg_colors[i]),
            hoverinfo='skip',
            showlegend=False,
        ))

    fig_cluster.add_trace(go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode='markers',
        marker=dict(
            size=3,
            color=mean_vel,
            colorscale='Turbo',
            cmin=vel_min,
            cmax=vel_max,
            showscale=False,
        ),
        name=f'Cluster {c} (n={n_c})',
        hovertemplate=(
            f'Cluster {c}<br>'
            'v_norm: %{marker.color:.3f}<br>'
            'x,y,z: (%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>'
        ),
    ))

# Shared colorbar for centroid velocity gradient.
fig_cluster.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode='markers',
    marker=dict(
        size=0.1,
        color=[vel_min],
        colorscale='Turbo',
        cmin=vel_min,
        cmax=vel_max,
        showscale=True,
        colorbar=dict(title='Centroid velocity (within-shot normalized)'),
    ),
    hoverinfo='skip',
    showlegend=False,
))

fig_cluster.update_layout(
    title='Catch-and-Shoot: Mean Aligned Curve by Cluster (Velocity Gradient)',
    scene=dict(
        xaxis_title='X (post-fit standardized)',
        yaxis_title='Y (post-fit standardized)',
        zaxis_title='Z (post-fit standardized)',
        aspectmode='data',
    ),
    height=760,
    margin=dict(l=0, r=0, t=60, b=0),
)
fig_cluster.show()

curves_df[['shot_id', 'player', 'shot_type', 'cluster']].head(12)

,shot_id,player,shot_type,cluster
0,18682,Player 133,Catch and Shoot,2
1,15059,Player 109,Catch and Shoot,0
2,19794,Player 145,Catch and Shoot,0
3,24755,Player 163,Catch and Shoot,1
4,13740,Player 102,Catch and Shoot,2
5,12074,Player 98,Catch and Shoot,0
6,16637,Player 118,Catch and Shoot,1
7,16798,Player 120,Catch and Shoot,1
8,17825,Player 125,Catch and Shoot,0
9,8947,Player 67,Catch and Shoot,1


In [13]:
# Off-the-dribble clustering (self-contained cell).
# Reuses traj_df from earlier pipeline and mirrors catch-and-shoot processing.

# Ensure geomstats compatibility in this environment.
if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
    np.trapz = np.trapezoid

import geomstats.backend as gs
from geomstats.geometry.discrete_curves import DiscreteCurvesStartingAtOrigin, SRVMetric
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

OD_PATTERN = 'off the dribble'
K_SAMPLING_OD = 60
MAX_CLUSTER_SHOTS_OD = 300
RANDOM_SEED_OD = 42
VELOCITY_WEIGHT_OD = 0.35
N_CLUSTERS_OD = 3

shot_meta_od = (
    traj_df[['shot_id', 'player', 'shot_type']]
    .drop_duplicates()
    .assign(shot_type_l=lambda x: x['shot_type'].astype(str).str.lower())
)

is_od = shot_meta_od['shot_type_l'].str.contains(OD_PATTERN, na=False)
off_ids = shot_meta_od.loc[is_od, 'shot_id'].tolist()
off_df = traj_df[traj_df['shot_id'].isin(off_ids)].copy()

print(f'Off-the-dribble shots available: {len(off_ids):,}')

curve_rows_od = []
for shot_id, g in off_df.groupby('shot_id', sort=False):
    g = g.sort_values('t_norm')

    t = g['t_norm'].to_numpy(dtype=float)
    x = g['x'].to_numpy(dtype=float)
    y = g['y'].to_numpy(dtype=float)
    z = g['z'].to_numpy(dtype=float)
    v = g['ball_velocity'].to_numpy(dtype=float)

    if len(t) < 5 or not np.isfinite(np.r_[t, x, y, z, v]).all():
        continue

    u = np.linspace(0.0, 1.0, len(t))
    u_dense = np.linspace(0.0, 1.0, K_SAMPLING_OD)

    x_dense = np.interp(u_dense, u, x)
    y_dense = np.interp(u_dense, u, y)
    z_dense = np.interp(u_dense, u, z)

    v_std = float(np.std(v))
    v_dense = np.interp(u_dense, u, v)
    if v_std > 1e-8:
        v_norm = (v_dense - float(np.mean(v))) / v_std
    else:
        v_norm = np.zeros_like(v_dense)

    meta = g[['player', 'shot_type']].iloc[0]
    curve_rows_od.append({
        'shot_id': int(shot_id),
        'player': meta['player'],
        'shot_type': meta['shot_type'],
        'curve_xyz': np.column_stack([x_dense, y_dense, z_dense]),
        'velocity_norm': v_norm,
    })

curves_df_od = pd.DataFrame(curve_rows_od)
if len(curves_df_od) > MAX_CLUSTER_SHOTS_OD:
    curves_df_od = curves_df_od.sample(MAX_CLUSTER_SHOTS_OD, random_state=RANDOM_SEED_OD).reset_index(drop=True)

if curves_df_od.empty:
    raise ValueError('No off-the-dribble trajectories available for clustering.')

curves_xyz_od = np.stack(curves_df_od['curve_xyz'].to_list(), axis=0)
vel_norm_od = np.stack(curves_df_od['velocity_norm'].to_list(), axis=0)

curves_r3_od = DiscreteCurvesStartingAtOrigin(
    ambient_dim=3,
    k_sampling_points=K_SAMPLING_OD,
    equip=False,
)

curves_proj_od = np.array(curves_r3_od.projection(gs.array(curves_xyz_od)))
curves_proj_od = np.array(curves_r3_od.normalize(gs.array(curves_proj_od)))

finite_mask_od = np.isfinite(curves_proj_od).all(axis=(1, 2))
if not finite_mask_od.all():
    curves_df_od = curves_df_od.loc[finite_mask_od].reset_index(drop=True)
    vel_norm_od = vel_norm_od[finite_mask_od]
    curves_proj_od = curves_proj_od[finite_mask_od]

curves_r3_od.equip_with_metric(SRVMetric)
curves_r3_od.equip_with_group_action('rotations')
curves_r3_od.equip_with_quotient()

template_od = gs.array(curves_proj_od[0])
aligned_curves_od = [np.array(template_od)]
align_failures_od = 0
for i in range(1, len(curves_proj_od)):
    p = gs.array(curves_proj_od[i])
    try:
        aligned = curves_r3_od.fiber_bundle.align(p, template_od)
        aligned_curves_od.append(np.array(aligned))
    except Exception:
        align_failures_od += 1
        aligned_curves_od.append(np.array(p))

aligned_curves_od = np.stack(aligned_curves_od, axis=0)

shape_features_od = aligned_curves_od.reshape(len(aligned_curves_od), -1)
feature_matrix_od = np.hstack([shape_features_od, VELOCITY_WEIGHT_OD * vel_norm_od])
feature_matrix_od = StandardScaler().fit_transform(feature_matrix_od)

kmeans_od = KMeans(n_clusters=N_CLUSTERS_OD, random_state=RANDOM_SEED_OD, n_init='auto')
labels_od = kmeans_od.fit_predict(feature_matrix_od)
curves_df_od['cluster'] = labels_od

print(f'Shots used for off-the-dribble clustering: {len(curves_df_od):,}')
print(f'Alignment fallbacks used: {align_failures_od}')
print('Cluster counts:')
print(curves_df_od['cluster'].value_counts().sort_index())

# Plot centroid curves with velocity gradient (same style as trajectory plots).
cluster_means_od = []
for c in sorted(curves_df_od['cluster'].unique()):
    idx = curves_df_od.index[curves_df_od['cluster'] == c].to_numpy()
    mean_curve = aligned_curves_od[idx].mean(axis=0)
    mean_vel = vel_norm_od[idx].mean(axis=0)
    cluster_means_od.append((c, mean_curve, mean_vel, len(idx)))

vel_all_od = np.concatenate([mvel for _, _, mvel, _ in cluster_means_od])
vel_min_od = float(np.min(vel_all_od))
vel_max_od = float(np.max(vel_all_od))

fig_od = go.Figure()
for c, mean_curve, mean_vel, n_c in cluster_means_od:
    x = mean_curve[:, 0]
    y = mean_curve[:, 1]
    z = mean_curve[:, 2]

    seg_vel = (mean_vel[:-1] + mean_vel[1:]) / 2.0
    seg_colors = velocity_to_color(seg_vel, vel_min_od, vel_max_od, colorscale='Turbo')

    for i in range(len(x) - 1):
        fig_od.add_trace(go.Scatter3d(
            x=[x[i], x[i + 1]],
            y=[y[i], y[i + 1]],
            z=[z[i], z[i + 1]],
            mode='lines',
            line=dict(width=9, color=seg_colors[i]),
            hoverinfo='skip',
            showlegend=False,
        ))

    fig_od.add_trace(go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode='markers',
        marker=dict(
            size=3,
            color=mean_vel,
            colorscale='Turbo',
            cmin=vel_min_od,
            cmax=vel_max_od,
            showscale=False,
        ),
        name=f'Cluster {c} (n={n_c})',
        hovertemplate=(
            f'Cluster {c}<br>'
            'v_norm: %{marker.color:.3f}<br>'
            'x,y,z: (%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>'
        ),
    ))

fig_od.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode='markers',
    marker=dict(
        size=0.1,
        color=[vel_min_od],
        colorscale='Turbo',
        cmin=vel_min_od,
        cmax=vel_max_od,
        showscale=True,
        colorbar=dict(title='Centroid velocity (within-shot normalized)'),
    ),
    hoverinfo='skip',
    showlegend=False,
))

fig_od.update_layout(
    title='Off-the-Dribble: Mean Aligned Curve by Cluster (Velocity Gradient)',
    scene=dict(
        xaxis_title='X (post-fit standardized)',
        yaxis_title='Y (post-fit standardized)',
        zaxis_title='Z (post-fit standardized)',
        aspectmode='data',
    ),
    height=760,
    margin=dict(l=0, r=0, t=60, b=0),
)
fig_od.show()

curves_df_od[['shot_id', 'player', 'shot_type', 'cluster']].head(12)

Off-the-dribble shots available: 13,575
Shots used for off-the-dribble clustering: 300
Alignment fallbacks used: 0
Cluster counts:
cluster
0    124
1     79
2     97
Name: count, dtype: int64


,shot_id,player,shot_type,cluster
0,1299,Player 15,Off the Dribble,2
1,24970,Player 163,Off the Dribble,2
2,21464,Player 158,Off the Dribble,2
3,5038,Player 52,Off the Dribble,0
4,15772,Player 116,Off the Dribble,2
5,22173,Player 158,Off the Dribble,0
6,20459,Player 153,Off the Dribble,0
7,5111,Player 52,Off the Dribble,0
8,21990,Player 158,Off the Dribble,2
9,5791,Player 59,Off the Dribble,2


## Off-the-Dribble Clustering with Geomstats

This repeats the same geometry-based clustering pipeline for off-the-dribble shots, including within-shot velocity normalization to reduce distance-driven speed effects.

In [8]:
# Geomstats alignment + clustering on catch-and-shoot trajectories.
if curves_df.empty:
    raise ValueError('No catch-and-shoot trajectories available for clustering.')

curves_xyz = np.stack(curves_df['curve_xyz'].to_list(), axis=0)
vel_norm = np.stack(curves_df['velocity_norm'].to_list(), axis=0)

curves_r3 = DiscreteCurvesStartingAtOrigin(
    ambient_dim=3,
    k_sampling_points=K_SAMPLING,
    equip=False,
)

curves_proj = np.array(curves_r3.projection(gs.array(curves_xyz)))
curves_proj = np.array(curves_r3.normalize(gs.array(curves_proj)))

# Keep only finite curves after projection/normalization.
finite_mask = np.isfinite(curves_proj).all(axis=(1, 2))
if not finite_mask.all():
    curves_df = curves_df.loc[finite_mask].reset_index(drop=True)
    vel_norm = vel_norm[finite_mask]
    curves_proj = curves_proj[finite_mask]

# Use SRV metric with rotation quotient for stable alignment on this dataset.
curves_r3.equip_with_metric(SRVMetric)
curves_r3.equip_with_group_action('rotations')
curves_r3.equip_with_quotient()

template = gs.array(curves_proj[0])
aligned_curves = [np.array(template)]
align_failures = 0
for i in range(1, len(curves_proj)):
    point = gs.array(curves_proj[i])
    try:
        aligned = curves_r3.fiber_bundle.align(point, template)
        aligned_curves.append(np.array(aligned))
    except Exception:
        align_failures += 1
        aligned_curves.append(np.array(point))

aligned_curves = np.stack(aligned_curves, axis=0)

shape_features = aligned_curves.reshape(len(aligned_curves), -1)
feature_matrix = np.hstack([shape_features, VELOCITY_WEIGHT * vel_norm])
feature_matrix = StandardScaler().fit_transform(feature_matrix)

N_CLUSTERS = 3
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init='auto')
cluster_labels = kmeans.fit_predict(feature_matrix)

curves_df['cluster'] = cluster_labels
print(f'Alignment fallbacks used: {align_failures}')
print('Cluster counts:')
print(curves_df['cluster'].value_counts().sort_index())
print('\nPlayers per cluster:')
print(curves_df.groupby('cluster')['player'].nunique())

Alignment fallbacks used: 0
Cluster counts:
cluster
0    106
1    113
2     81
Name: count, dtype: int64

Players per cluster:
cluster
0    32
1    43
2    40
Name: player, dtype: int64


In [7]:
# Prepare fixed-length catch-and-shoot trajectories and within-shot velocity normalization.
CATCH_PATTERNS = ['catch and shoot', 'catch & shoot', 'catch-and-shoot']
K_SAMPLING = 60
MAX_CLUSTER_SHOTS = 300
RANDOM_SEED = 42
VELOCITY_WEIGHT = 0.35

shot_meta = (
    traj_df[['shot_id', 'player', 'shot_type']]
    .drop_duplicates()
    .assign(shot_type_l=lambda x: x['shot_type'].astype(str).str.lower())
)

is_cns = shot_meta['shot_type_l'].apply(
    lambda s: any(p in s for p in CATCH_PATTERNS)
)
catch_ids = shot_meta.loc[is_cns, 'shot_id'].tolist()

catch_df = traj_df[traj_df['shot_id'].isin(catch_ids)].copy()
print(f'Catch-and-shoot shots available: {len(catch_ids):,}')

curve_rows = []
for shot_id, g in catch_df.groupby('shot_id', sort=False):
    g = g.sort_values('t_norm')

    t = g['t_norm'].to_numpy(dtype=float)
    x = g['x'].to_numpy(dtype=float)
    y = g['y'].to_numpy(dtype=float)
    z = g['z'].to_numpy(dtype=float)
    v = g['ball_velocity'].to_numpy(dtype=float)

    if len(t) < 5 or not np.isfinite(np.r_[t, x, y, z, v]).all():
        continue

    # Reparameterize by normalized progress to make every shot same sampling length.
    u = np.linspace(0.0, 1.0, len(t))
    u_dense = np.linspace(0.0, 1.0, K_SAMPLING)

    x_dense = np.interp(u_dense, u, x)
    y_dense = np.interp(u_dense, u, y)
    z_dense = np.interp(u_dense, u, z)

    # Normalize velocity *within shot* to reduce distance-driven speed magnitude effects.
    v_std = float(np.std(v))
    v_dense = np.interp(u_dense, u, v)
    if v_std > 1e-8:
        v_norm = (v_dense - float(np.mean(v))) / v_std
    else:
        v_norm = np.zeros_like(v_dense)

    meta = g[['player', 'shot_type']].iloc[0]
    curve_rows.append({
        'shot_id': int(shot_id),
        'player': meta['player'],
        'shot_type': meta['shot_type'],
        'curve_xyz': np.column_stack([x_dense, y_dense, z_dense]),
        'velocity_norm': v_norm,
    })

curves_df = pd.DataFrame(curve_rows)

if len(curves_df) > MAX_CLUSTER_SHOTS:
    curves_df = curves_df.sample(MAX_CLUSTER_SHOTS, random_state=RANDOM_SEED).reset_index(drop=True)

print(f'Shots used for geomstats clustering: {len(curves_df):,}')

Catch-and-shoot shots available: 12,230
Shots used for geomstats clustering: 300


In [6]:
# Compatibility shim: recent numpy versions expose trapezoid, while geomstats expects trapz.
if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
    np.trapz = np.trapezoid

import geomstats.backend as gs
from geomstats.geometry.discrete_curves import DiscreteCurvesStartingAtOrigin, SRVMetric
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print('Geomstats + sklearn clustering imports loaded.')

Geomstats + sklearn clustering imports loaded.


## Notes

- Phase-time mapping uses available timing columns to preserve the requested order `PreHitch -> Hitch -> PostHitch -> Release`.

- Interpolation uses a shape-preserving method and strict time monotonicity checks to avoid overshoot artifacts.

- Coordinates are translated so `PreHitch` is at the origin using `BallXPreHitch`, `BallYPreHitch`, and `BallZPreHitch`.

- Each shot is then rotated in XY so the pre-hitch-to-basket direction is always aligned with +Y.

- After spline fitting, x/y/z are standardized within player using axis-specific standard deviation (toggle with `POST_STANDARDIZATION`).

- Curves are rendered as short line segments colored by interpolated velocity to achieve along-path velocity coloring.

## Cluster Characteristics Analysis (Within Shot Type)

This section summarizes domain-specific trajectory characteristics **within each shot type** cluster using the aligned curves and within-shot normalized velocity profiles.

In [14]:
from IPython.display import display


def build_cluster_profile(curves_df_local, aligned_curves_local, vel_norm_local, label):
    rows = []
    n_pts = aligned_curves_local.shape[1]
    split = max(2, n_pts // 3)

    for i in range(len(curves_df_local)):
        c = aligned_curves_local[i]
        v = vel_norm_local[i]

        x = c[:, 0]
        y = c[:, 1]
        z = c[:, 2]

        seg = np.diff(c, axis=0)
        path_length = float(np.sum(np.linalg.norm(seg, axis=1)))

        rows.append({
            'cluster': int(curves_df_local.iloc[i]['cluster']),
            'shot_id': int(curves_df_local.iloc[i]['shot_id']),
            'player': curves_df_local.iloc[i]['player'],
            'forward_carry': float(y[-1] - y[0]),
            'lateral_span': float(np.max(np.abs(x))),
            'lateral_finish': float(x[-1] - x[0]),
            'vertical_lift': float(np.max(z) - z[0]),
            'release_height_norm': float(z[-1]),
            'path_length': path_length,
            'early_velocity_norm': float(np.mean(v[:split])),
            'late_velocity_norm': float(np.mean(v[-split:])),
            'velocity_ramp': float(np.mean(v[-split:]) - np.mean(v[:split])),
        })

    per_shot = pd.DataFrame(rows)

    profile = (
        per_shot.groupby('cluster')
        .agg(
            n_shots=('shot_id', 'count'),
            n_players=('player', 'nunique'),
            forward_carry=('forward_carry', 'mean'),
            lateral_span=('lateral_span', 'mean'),
            lateral_finish=('lateral_finish', 'mean'),
            vertical_lift=('vertical_lift', 'mean'),
            release_height_norm=('release_height_norm', 'mean'),
            path_length=('path_length', 'mean'),
            early_velocity_norm=('early_velocity_norm', 'mean'),
            late_velocity_norm=('late_velocity_norm', 'mean'),
            velocity_ramp=('velocity_ramp', 'mean'),
        )
        .reset_index()
        .sort_values('cluster')
    )

    print(f'\n=== {label}: Cluster Profile ===')
    display(profile.round(4))

    # Rank clusters for quick interpretation.
    rank_cols = ['forward_carry', 'lateral_span', 'vertical_lift', 'path_length', 'velocity_ramp']
    rank_table = profile[['cluster'] + rank_cols].copy()
    for col in rank_cols:
        rank_table[col + '_rank'] = rank_table[col].rank(ascending=False, method='dense').astype(int)

    print(f'=== {label}: Relative Ranking (1 = highest) ===')
    display(rank_table[['cluster'] + [c + '_rank' for c in rank_cols]])

    return per_shot, profile


# Catch-and-shoot profiles
if {'curves_df', 'aligned_curves', 'vel_norm'} <= set(globals().keys()):
    cns_per_shot, cns_profile = build_cluster_profile(curves_df, aligned_curves, vel_norm, 'Catch-and-Shoot')
else:
    print('Catch-and-shoot clustering variables not found. Re-run the catch-and-shoot clustering cells first.')

# Off-the-dribble profiles
if {'curves_df_od', 'aligned_curves_od', 'vel_norm_od'} <= set(globals().keys()):
    od_per_shot, od_profile = build_cluster_profile(curves_df_od, aligned_curves_od, vel_norm_od, 'Off-the-Dribble')
else:
    print('Off-the-dribble clustering variables not found. Re-run the off-the-dribble clustering cell first.')


=== Catch-and-Shoot: Cluster Profile ===


,cluster,n_shots,n_players,forward_carry,lateral_span,lateral_finish,vertical_lift,release_height_norm,path_length,early_velocity_norm,late_velocity_norm,velocity_ramp
0,0,106,32,-0.0063,0.1869,0.0370,0.5267,0.5331,0.8946,-0.7220,0.9768,1.6988
1,1,113,43,0.0942,0.1275,0.0395,0.4976,0.5061,0.9007,-0.7336,0.6547,1.3884
2,2,81,40,0.0422,0.1298,0.1005,0.3684,0.3700,0.7393,-1.1655,0.7187,1.8841


=== Catch-and-Shoot: Relative Ranking (1 = highest) ===


,cluster,forward_carry_rank,lateral_span_rank,vertical_lift_rank,path_length_rank,velocity_ramp_rank
0,0,3,1,1,2,2
1,1,1,3,2,1,3
2,2,2,2,3,3,1



=== Off-the-Dribble: Cluster Profile ===


,cluster,n_shots,n_players,forward_carry,lateral_span,lateral_finish,vertical_lift,release_height_norm,path_length,early_velocity_norm,late_velocity_norm,velocity_ramp
0,0,124,53,0.0630,0.2460,0.2478,0.4196,0.4352,0.8891,-0.6926,0.5885,1.2810
1,1,79,49,0.1314,0.1667,0.1225,0.2726,0.2788,0.7154,-1.1159,0.6120,1.7279
2,2,97,32,0.0149,0.2930,0.2664,0.4879,0.4980,0.8827,-0.5447,0.8848,1.4295


=== Off-the-Dribble: Relative Ranking (1 = highest) ===


,cluster,forward_carry_rank,lateral_span_rank,vertical_lift_rank,path_length_rank,velocity_ramp_rank
0,0,2,2,2,1,3
1,1,1,3,3,3,1
2,2,3,1,1,2,2


In [16]:
def summarize_cluster_characteristics(profile_df, label):
    if profile_df is None or profile_df.empty:
        return [f'- {label}: no cluster profile available.']

    cols = ['forward_carry', 'lateral_span', 'vertical_lift', 'path_length', 'velocity_ramp']
    valid_cols = [c for c in cols if c in profile_df.columns]

    lines = [f'### {label}']

    for col in valid_cols:
        high_idx = profile_df[col].idxmax()
        low_idx = profile_df[col].idxmin()
        high_row = profile_df.loc[high_idx]
        low_row = profile_df.loc[low_idx]

        lines.append(
            f"- Highest {col.replace('_', ' ')}: Cluster {int(high_row['cluster'])} "
            f"({high_row[col]:.3f}) vs lowest Cluster {int(low_row['cluster'])} ({low_row[col]:.3f})."
        )

    return lines

analysis_lines = ['## Domain-Specific Cluster Interpretation']

if 'cns_profile' in globals():
    analysis_lines.extend(summarize_cluster_characteristics(cns_profile, 'Catch-and-Shoot'))

if 'od_profile' in globals():
    analysis_lines.extend(summarize_cluster_characteristics(od_profile, 'Off-the-Dribble'))

analysis_lines.extend([
    '### Practical Read of the Clusters',
    '- Clusters with higher forward carry and lower lateral span generally indicate more direct ball transfer into release.',
    '- Clusters with higher vertical lift and longer path length suggest more pronounced loading patterns before release.',
    '- Positive velocity ramp indicates acceleration into release; flatter or negative ramp can indicate earlier speed generation or deceleration near release.',
])

print('\n'.join(analysis_lines))

## Domain-Specific Cluster Interpretation
### Catch-and-Shoot
- Highest forward carry: Cluster 1 (0.094) vs lowest Cluster 0 (-0.006).
- Highest lateral span: Cluster 0 (0.187) vs lowest Cluster 1 (0.128).
- Highest vertical lift: Cluster 0 (0.527) vs lowest Cluster 2 (0.368).
- Highest path length: Cluster 1 (0.901) vs lowest Cluster 2 (0.739).
- Highest velocity ramp: Cluster 2 (1.884) vs lowest Cluster 1 (1.388).
### Off-the-Dribble
- Highest forward carry: Cluster 1 (0.131) vs lowest Cluster 2 (0.015).
- Highest lateral span: Cluster 2 (0.293) vs lowest Cluster 1 (0.167).
- Highest vertical lift: Cluster 2 (0.488) vs lowest Cluster 1 (0.273).
- Highest path length: Cluster 0 (0.889) vs lowest Cluster 1 (0.715).
- Highest velocity ramp: Cluster 1 (1.728) vs lowest Cluster 0 (1.281).
### Practical Read of the Clusters
- Clusters with higher forward carry and lower lateral span generally indicate more direct ball transfer into release.
- Clusters with higher vertical lift and longer p

## Domain-Specific Cluster Characteristics (Polished Narrative)

Within shot type, the clustering separates distinct movement signatures in how the ball is loaded and delivered into release.

### Catch-and-Shoot
Cluster 1 represents the most direct forward transfer profile. It has the largest average forward carry and the longest average pre-release path, while maintaining the smallest lateral spread. This pattern is consistent with an efficient gather where the ball advances toward release with limited side-to-side deviation.

Cluster 0 captures a higher-lift, wider-sweep pattern. It shows the greatest vertical rise and the largest lateral span, indicating a more pronounced upward and side-to-side loading action before release.

Cluster 2 is the compact, late-acceleration profile. It has the shortest average path and lowest vertical lift, but the strongest velocity ramp into release. In practice, this looks like a tighter pre-release trajectory that speeds up more aggressively near the end of the motion.

### Off-the-Dribble
Cluster 1 is the drive-forward acceleration profile. It has the highest forward carry and strongest velocity ramp, but the lowest lift, shortest path, and smallest lateral span. This indicates a compact dribble-to-shot transfer that emphasizes late speed generation.

Cluster 2 is the high-lift lateral-creation profile. It has the largest lateral span and highest vertical lift, suggesting greater ball repositioning and upward loading before release.

Cluster 0 is the long-path control profile. It has the longest average path length with moderate forward carry and lift, consistent with a more extended but controlled pre-release route.

### Basketball Interpretation
These clusters likely reflect different functional shooting strategies rather than simple good/bad categories:

- Direct-transfer profiles emphasize line-to-target efficiency and reduced lateral movement.
- High-lift/lateral profiles may reflect added creation space, balance correction, or timing control before release.
- Compact late-ramp profiles emphasize acceleration near release and may support faster release timing.

Because clustering was run separately by shot type, these interpretations should be used for within-type comparisons (cluster vs cluster inside catch-and-shoot, and cluster vs cluster inside off-the-dribble), not for direct cross-type ranking.